In [ ]:
import tensorflow as tf
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from collections import Counter

# 1. Pengaturan dasar
DATA_DIR = Path("flowers")
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

# 2. Membaca semua gambar dan label
class_names = sorted([
    folder.name for folder in DATA_DIR.iterdir()
    if folder.is_dir()
])

class_to_index = {
    name: index for index, name in enumerate(class_names)
}

image_paths = []
labels = []

for class_name in class_names:
    files = list((DATA_DIR / class_name).glob("*.jpg"))

    for file in files:
        image_paths.append(str(file))
        labels.append(class_to_index[class_name])

image_paths = np.array(image_paths)
labels = np.array(labels)

print("Daftar kelas:", class_names)
print("Total gambar:", len(image_paths))

# 3. Membagi dataset: 70% train, 15% validation, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    image_paths,
    labels,
    test_size=0.30,
    random_state=SEED,
    stratify=labels
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

print("\nJumlah data:")
print("Training  :", len(X_train))
print("Validation:", len(X_val))
print("Testing   :", len(X_test))

# 4. Fungsi membaca dan mengubah ukuran gambar
def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)

    # Format float32, rentang piksel 0-255
    image = tf.cast(image, tf.float32)

    return image, label

# 5. Membuat pipeline dataset
def create_dataset(paths, labels, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))

    dataset = dataset.map(
        load_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(paths),
            seed=SEED
        )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

train_ds = create_dataset(X_train, y_train, training=True)
val_ds = create_dataset(X_val, y_val)
test_ds = create_dataset(X_test, y_test)

# 6. Augmentasi khusus training
def create_augmentation():
    return tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.1),
        tf.keras.layers.RandomZoom(0.1),
        tf.keras.layers.RandomContrast(0.1)
    ], name="data_augmentation")

print("\nPreprocessing selesai!")
